# Hubmicroo Voice Assistant — Kaggle Test Environment
> **Testing only** — sessions expire after 12 h. For production use `docker compose up`.

**Before running:** Enable **GPU T4 x2** + **Internet** in Kaggle Settings.

In [ ]:
# ── Step 1: System dependencies ───────────────────────────────────────────
import subprocess, sys

def run(cmd, check=False):
    print(f'$ {cmd[:100]}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-500:])
    if r.returncode != 0 and check:
        raise RuntimeError(f'Command failed: {r.stderr[-300:]}')
    return r

run('apt-get update -qq && apt-get install -y --no-install-recommends ffmpeg libgomp1 curl wget tar', check=True)
print('System deps OK')

In [ ]:
# ── Step 2: Python packages ────────────────────────────────────────────────
# Install in correct order to avoid dependency conflicts
run('pip install -q torch --index-url https://download.pytorch.org/whl/cu121')
run('pip install -q fastapi "uvicorn[standard]" pydantic pydantic-settings python-multipart')
run('pip install -q "python-jose[cryptography]" "passlib[bcrypt]" bcrypt')
run('pip install -q qdrant-client rank-bm25 httpx')
run('pip install -q FlagEmbedding transformers sentence-transformers')
run('pip install -q faster-whisper')
run('pip install -q pyngrok requests')
print('\n✅ All packages installed')

In [ ]:
# ── Step 3: Start Qdrant ───────────────────────────────────────────────────
import subprocess, time, os, urllib.request, tarfile, pathlib, json

qdrant_bin = pathlib.Path('/usr/local/bin/qdrant')
if not qdrant_bin.exists():
    print('Downloading Qdrant...')
    url = 'https://github.com/qdrant/qdrant/releases/download/v1.13.6/qdrant-x86_64-unknown-linux-musl.tar.gz'
    urllib.request.urlretrieve(url, '/tmp/qdrant.tar.gz')
    with tarfile.open('/tmp/qdrant.tar.gz') as t:
        t.extract('qdrant', '/usr/local/bin')
    os.chmod('/usr/local/bin/qdrant', 0o755)
    print('Qdrant downloaded')

os.makedirs('/tmp/qdrant_storage', exist_ok=True)
qdrant_proc = subprocess.Popen(
    ['/usr/local/bin/qdrant', '--storage-path', '/tmp/qdrant_storage'],
    stdout=open('/tmp/qdrant.log','w'), stderr=subprocess.STDOUT
)
print('Waiting for Qdrant...')
for _ in range(20):
    try:
        urllib.request.urlopen('http://localhost:6333/healthz', timeout=2)
        print('✅ Qdrant ready')
        break
    except:
        time.sleep(2)
else:
    raise RuntimeError('Qdrant failed to start. Check /tmp/qdrant.log')

In [ ]:
# ── Step 4: Install Ollama and pull model ─────────────────────────────────
import subprocess, time, os, urllib.request, pathlib

if not pathlib.Path('/usr/local/bin/ollama').exists():
    print('Installing Ollama...')
    r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('Ollama install stderr:', r.stderr[-500:])
    print('Ollama installed')

env = os.environ.copy()
env['OLLAMA_KEEP_ALIVE'] = '5m'
env['HOME'] = '/root'
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'], env=env,
    stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT
)
print('Waiting for Ollama server...')
for _ in range(20):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print('✅ Ollama server ready')
        break
    except:
        time.sleep(3)
else:
    raise RuntimeError('Ollama failed to start. Check /tmp/ollama.log')

MODEL = os.environ.get('LLM_MODEL', 'qwen3:4b')
print(f'Pulling {MODEL} (several minutes on first run)...')
r = subprocess.run(f'ollama pull {MODEL}', shell=True, capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else 'done')
print('✅ Model ready')

In [ ]:
# ── Step 5: Clone repo & configure environment ────────────────────────────
import sys, os, pathlib, subprocess

REPO_ROOT = pathlib.Path('/kaggle/working/Hubmicroo_VoiceAssistant')

if not REPO_ROOT.exists():
    print('Cloning repo...')
    r = subprocess.run(
        'git clone https://github.com/ai-with-abdullah/Hubmicroo-AI-Voice-Assistant.git '
        '/kaggle/working/Hubmicroo_VoiceAssistant',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed: {r.stderr}')
    print('✅ Repo cloned')
else:
    print('Repo already present, pulling latest...')
    subprocess.run('git -C /kaggle/working/Hubmicroo_VoiceAssistant pull', shell=True)

# Add to Python path
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

DATA_DIR = str(REPO_ROOT / 'backend' / 'data')
FRONTEND_DIR = str(REPO_ROOT / 'frontend')

env_content = f"""LLM_MODEL=qwen3:4b
OLLAMA_BASE_URL=http://localhost:11434
QDRANT_URL=http://localhost:6333
EMBED_MODEL=BAAI/bge-m3
EMBED_DEVICE=cuda
WHISPER_DEVICE=cuda
WHISPER_MODEL=base
TTS_ENABLED=false
DATA_DIR={DATA_DIR}
PRODUCTS_FILE={DATA_DIR}/products.json
PAGES_FILE={DATA_DIR}/pages.json
FRONTEND_DIR={FRONTEND_DIR}
ADMIN_USERNAME=admin
ADMIN_PASSWORD=kaggle-test-pass
SECRET_KEY=kaggle-testing-secret-32-chars-ok
CORS_ORIGINS=*
"""
(REPO_ROOT / '.env').write_text(env_content)
print('✅ .env written')
print('Data dir:', DATA_DIR)
print('Products file exists:', (pathlib.Path(DATA_DIR) / 'products.json').exists())

In [ ]:
# ── Step 6: Seed the vector index ─────────────────────────────────────────
# Reset settings cache so it picks up the new .env
import importlib, sys

# Clear cached settings if already loaded
for mod in list(sys.modules.keys()):
    if 'backend' in mod or 'hubmicroo' in mod.lower():
        del sys.modules[mod]

from backend.app.config import get_settings
from backend.app import store
from backend.app.indexer import rebuild_all

s = get_settings()
print('DATA_DIR:', s.DATA_DIR)
print('QDRANT_URL:', s.QDRANT_URL)

store.ensure_collection()
count = rebuild_all()
print(f'✅ Index seeded: {count} items')

In [ ]:
# ── Step 7: Start FastAPI backend ─────────────────────────────────────────
import subprocess, time, urllib.request, json, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/Hubmicroo_VoiceAssistant'

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.app.main:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd='/kaggle/working/Hubmicroo_VoiceAssistant',
    env=env,
    stdout=open('/tmp/fastapi.log', 'w'),
    stderr=subprocess.STDOUT
)

print('Waiting for backend (up to 90s for BGE-M3 load)...')
for i in range(30):
    time.sleep(3)
    try:
        with urllib.request.urlopen('http://localhost:8000/api/health', timeout=5) as r:
            h = json.loads(r.read())
        print(f'✅ Backend ready: {h}')
        break
    except Exception as e:
        if i % 5 == 0:
            print(f'  [{i*3}s] still waiting... ({e})')
else:
    print('❌ Backend not responding. Last log lines:')
    with open('/tmp/fastapi.log') as f:
        print(''.join(f.readlines()[-20:]))

In [ ]:
# ── Step 8: Expose via ngrok ──────────────────────────────────────────────
# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # <── paste your token here for a public URL

from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
else:
    public_url = 'http://localhost:8000 (no ngrok — use port-forwarding or add a token)'

print('\n' + '='*65)
print(f'  🛍️  Store widget  : {public_url}/')
print(f'  🔧  Admin panel   : {public_url}/admin  (pass: kaggle-test-pass)')
print(f'  📖  API docs      : {public_url}/docs')
print(f'  ❤️   Health check  : {public_url}/api/health')
print('='*65 + '\n')

In [ ]:
# ── Step 9: Quick smoke test ──────────────────────────────────────────────
import urllib.request, json

queries = [
    ('en', 'Do you have wireless headphones under 5000 PKR?'),
    ('ur', 'kya aap ke paas wireless headphones hain?'),
    ('ar', 'هل لديكم سماعات بلوتوث لاسلكية؟'),
    ('en', 'What is your return policy?'),
]

for lang, q in queries:
    payload = json.dumps({'message': q}).encode()
    req = urllib.request.Request(
        'http://localhost:8000/api/chat',
        data=payload,
        headers={'Content-Type': 'application/json'}
    )
    try:
        with urllib.request.urlopen(req, timeout=90) as r:
            resp = json.loads(r.read())
        print(f'[{lang.upper()}] Q: {q}')
        print(f'       A: {resp["answer"][:120]}')
        products = [p["name"] for p in resp.get("products", [])]
        if products:
            print(f'  🛍️  Products: {products}')
        print(f'  🌐 Lang={resp["lang"]} | Cached={resp["cached"]}')
        print()
    except Exception as e:
        print(f'[{lang.upper()}] ERROR: {e}\n')

In [ ]:
# ── Step 10: Run eval harness ─────────────────────────────────────────────
import subprocess

result = subprocess.run(
    ['python', 'eval/run_eval.py',
     '--base-url', 'http://localhost:8000',
     '--out', '/tmp/eval_results.json'],
    cwd='/kaggle/working/Hubmicroo_VoiceAssistant',
    capture_output=True, text=True, timeout=600
)
print(result.stdout)
if result.returncode != 0 and result.stderr:
    print('STDERR:', result.stderr[-500:])